In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [16]:
# Installing local embedding libraries to bypass OpenAI quota issue
!pip install -q --no-warn-conflicts chromadb langchain-chroma langchain-community langchain-text-splitters sentence-transformers

In [17]:
import os
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✓ Local setup ready. No OpenAI API Key needed now!")

✓ Local setup ready. No OpenAI API Key needed now!


In [14]:
import os

# Create directory
os.makedirs('company_docs', exist_ok=True)

# Generate sample files
sample_docs = {
    "vacation_policy.txt": "Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance.",
    "wfh_policy.txt": "Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval.",
    "maternity_policy.txt": "The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers."
}

for filename, content in sample_docs.items():
    with open(f"company_docs/{filename}", "w") as f:
        f.write(content)

print("✓ Setup complete: Sample documents generated in 'company_docs/' folder.")

✓ Setup complete: Sample documents generated in 'company_docs/' folder.


Cell 4: Task 1.1 – Generate Embeddings

In [19]:
from sentence_transformers import SentenceTransformer

# Load a completely free, open-source embedding model
local_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    """Generate embedding for text using local Hugging Face model. Returns: List of numbers."""
    embedding = local_model.encode(text)
    return embedding.tolist()

# Test the embedding function
text = 'vacation policy'
embedding = get_embedding(text)
print(f'Embedding length: {len(embedding)} (Using local 384-dim model)')
print(f'First 5 values: {embedding[:5]}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding length: 384 (Using local 384-dim model)
First 5 values: [0.04388044402003288, 0.058420415967702866, 0.08754625916481018, -0.000251249031862244, 0.0365716777741909]


Cell 5: Task 1.2 – Calculate Similarity

In [20]:
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    
    dot_product = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    
    return dot_product / (norm1 * norm2)

# Test with similar and dissimilar phrases
phrases = ['vacation policy', 'time off rules', 'PTO guidelines', 'dress code requirements']
embeddings = [get_embedding(p) for p in phrases]

base_embedding = embeddings[0]
print(f'Comparing "{phrases[0]}" with:\n')

for i, phrase in enumerate(phrases[1:], start=1):
    similarity = cosine_similarity(base_embedding, embeddings[i])
    print(f'{phrase:30} Similarity: {similarity:.4f}')

Comparing "vacation policy" with:

time off rules                 Similarity: 0.2653
PTO guidelines                 Similarity: 0.1812
dress code requirements        Similarity: 0.1095


Cell 6: Task 2.1 – Initialize ChromaDB (Local Version)

In [21]:
import chromadb
from chromadb.utils import embedding_functions

# Create ChromaDB persistent client
chroma_client = chromadb.PersistentClient(path='./chromadb')

# Use ChromaDB's built-in default embedding function (Sentence Transformers)
# This works 100% locally and does NOT require any API key!
default_ef = embedding_functions.DefaultEmbeddingFunction()

# Get or create collection
collection = chroma_client.get_or_create_collection(
    name='company_docs',
    embedding_function=default_ef,
    metadata={'description': 'Company policy documents'}
)

print(f'Collection Name: {collection.name}')
print(f'Current Document Count: {collection.count()}')

Collection Name: company_docs
Current Document Count: 0


Cell 7: Task 2.2 – Load and Index Documents

In [22]:
# Load text documents from the folder
loader = DirectoryLoader('company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()

# Split into chunks of 500 characters
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Add to ChromaDB if empty
if collection.count() == 0:
    collection.add(
        documents=[chunk.page_content for chunk in chunks],
        ids=[f'doc_{i}' for i in range(len(chunks))],
        metadatas=[{'source': chunk.metadata.get('source', 'unknown')} for chunk in chunks]
    )
    print(f'✓ Successfully added {len(chunks)} chunks to ChromaDB.')
else:
    print(f'Collection already contains {collection.count()} documents. Skipping insertion.')

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 99.5MiB/s]


✓ Successfully added 3 chunks to ChromaDB.


Cell 8: Task 3.1 – Test Vector Search

In [23]:
def vector_search(query, n_results=3):
    """Search ChromaDB collection using semantic similarity."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test queries
test_queries = ['time off policy', 'WFH guidelines', 'maternity leave']

for query in test_queries:
    print(f'\n{"="*60}')
    print(f'Query: {query}')
    print(f'{"="*60}')
    
    results = vector_search(query, n_results=2)
    
    for i, doc in enumerate(results['documents'][0]):
        distance = results['distances'][0][i]
        print(f'\nResult {i+1} (Distance/Score: {distance:.4f}):')
        print(doc[:200] + '...')


Query: time off policy

Result 1 (Distance/Score: 1.0252):
Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance....

Result 2 (Distance/Score: 1.3234):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Query: WFH guidelines

Result 1 (Distance/Score: 1.2676):
Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval....

Result 2 (Distance/Score: 1.8423):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Query: maternity leave

Result 1 (Distance/Score: 0.6504):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Result 2 (Distance/Score: 1.5459):
Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval....


Cell 9: Task 3.2 – Build Semantic RAG Pipeline (Optional Integration)

In [24]:
from langchain_openai import ChatOpenAI

# Initialize LLM (Requires valid API key for generation)
try:
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
except Exception as e:
    llm = None

def semantic_rag(query, n_results=3):
    """Complete semantic RAG pipeline."""
    # Step 1: Vector search (Local & Free)
    results = vector_search(query, n_results=n_results)
    
    if not results['documents'] or not results['documents'][0]:
        return 'No relevant information found.'
    
    # Step 2: Build context
    context = '\n\n---\n\n'.join(results['documents'][0])
    
    # Step 3: Create prompt
    prompt = f"""You are a helpful HR assistant. Answer using ONLY the context below. If not in context, say so.

Context:
{context}

Question: {query}
Answer:"""
    
    # Step 4: Generate text via LLM
    if llm is not None:
        try:
            messages = [{'role': 'user', 'content': prompt}]
            response = llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"[Context Retrieved Successfully! But LLM Generation failed due to Quota/API Error]\nContext Found:\n{context}"
    else:
        return f"[LLM not initialized] Context Found:\n{context}"

# Run QA test
questions = [
    'How much time off do employees get?',
    'Can I work from home?',
    'What is the maternity leave policy?'
]

for q in questions:
    print(f'\nQ: {q}')
    print(f'A: {semantic_rag(q)}')


Q: How much time off do employees get?
A: [Context Retrieved Successfully! But LLM Generation failed due to Quota/API Error]
Context Found:
Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance.

---

Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval.

---

The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers.

Q: Can I work from home?
A: [Context Retrieved Successfully! But LLM Generation failed due to Quota/API Error]
Context Found:
Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval.

---

The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers.

---

Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance.

Q: What is the maternity leave policy?
A: [Context Retrieved Succes

Ce****ll 10: Bonus – Keyword vs Semantic Comparison****

In [25]:
def keyword_search(query, chunks, top_k=3):
    """Simple keyword frequency counter."""
    scored = []
    query_lower = query.lower()
    words = query_lower.split()
    
    for chunk in chunks:
        score = sum(chunk.page_content.lower().count(word) for word in words)
        if score > 0:
            scored.append((score, chunk))
            
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for s, c in scored[:top_k]]

# Run Test
query = 'PTO policy'

print('--- KEYWORD SEARCH ---')
kw_results = keyword_search(query, chunks, top_k=2)
print(f'Found: {len(kw_results)} results')

print('\n--- SEMANTIC SEARCH ---')
sem_results = vector_search(query, n_results=2)
print(f'Found: {len(sem_results["documents"][0])} results')
if len(sem_results["documents"][0]) > 0:
    print(f'Top result: {sem_results["documents"][0][0][:120]}...')

--- KEYWORD SEARCH ---
Found: 2 results

--- SEMANTIC SEARCH ---
Found: 2 results
Top result: Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance....
